# Import

In [1]:
import os
import sys

import numpy as  np
from lwm_model import lwm  # 클래스 import
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time

from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

# Setting 
### CUDA 몇번 사용하는지 잘 확인하기

In [ ]:
# Scenes 1000
## Subcarriers 64

scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

params = param_manager.get_params()

param_manager.params["scenes"] =list(range(1000))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")


현재 사용 중인 장치: cuda:2


# Generate a dataset

In [3]:
# scenes / subcarriers는 그대로
param_manager.params["radar"]["enable"] = False

# lidar가 dict로 존재하면 disable
if isinstance(param_manager.params.get("lidar", None), dict):
    param_manager.params["lidar"]["enable"] = False

# position이 dict로 존재하면 disable
if isinstance(param_manager.params.get("position", None), dict):
    param_manager.params["position"]["enable"] = False

# Radar, LiDAR, Position 사용하지 않으므로 False 
dataset = Dataset(param_manager)

Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (8.75s)


# Prerocessing

#### Channel

In [4]:
def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]

    # 케이스1) list/tuple이면 ue_idx로 선택
    if isinstance(ue_obj, (list, tuple)):
        ch_obj = ue_obj[ue_idx]
    else:
        # 케이스2) 단일 OFDMChannel이면 그대로 사용
        ch_obj = ue_obj

    # coeffs는 dict key가 아니라 attribute일 확률이 매우 큼
    if hasattr(ch_obj, "coeffs"):
        return ch_obj.coeffs

    # 혹시 dict라면 마지막 보험
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]

    raise TypeError(f"Cannot get coeffs. ue type={type(ue_obj)}, ch type={type(ch_obj)}")


In [5]:
def get_train_min_max_realimag(frames, train_idx, us_idx=0):

    rmin, rmax =  float('inf'), float('-inf')
    imin, imax =  float('inf'), float('-inf')

    print("Calculating min/max over training set...")

    for t  in train_idx:
        frame  = frames[t]
        cooeffs  = get_coeffs_from_frame(frame, us_idx)  # (N_subcarriers, )

        rmin = min(rmin, float(cooeffs.real.min()))
        rmax = max(rmax, float(cooeffs.real.max()))
        imin = min(imin, float(cooeffs.imag.min()))
        imax = max(imax, float(cooeffs.imag.max()))

    print(f"Done. rmin={rmin}, rmax={rmax}, imin={imin}, imax={imax}")
    return (rmin, rmax), (imin, imax)

In [6]:
def preprocess_channel_coeffs_minmax(coeffs_np, r_min, r_max, i_min, i_max, device=device, eps=1e-16):
    # Convert Numpy to Tensor
    coeffs = torch.from_numpy(coeffs_np).to(torch.complex64)
    
    
    r = coeffs.real
    i = coeffs.imag
    
    # Min-Max Scaling [0, 1]
    # Add eps to denominator to prevent division by zero
    r_scaled = (r - r_min) / max(r_max - r_min, eps)
    i_scaled = (i - i_min) / max(i_max - i_min, eps)
    
    # Concat (Maintains shape like (..., 2*subcarriers))
    H = torch.cat([r_scaled, i_scaled], dim=-1).to(device, dtype=torch.float32)
    return H

#### image

# Dataset 구현

In [7]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

class ChannelNextStepDatasetGPU(TorchDataset):
    def __init__(self, comm_frames, ue_idx=0, past_len=16, device=device,
                 r_min=0.0, r_max=1.0, i_min=0.0, i_max=1.0):
        self.comm_frames = comm_frames
        self.ue_idx = ue_idx
        self.past_len = past_len
        self.device = device

        self.r_min, self.r_max = r_min, r_max
        self.i_min, self.i_max = i_min, i_max

        self.N = len(self.comm_frames)
        self.valid_start = past_len - 1
        self.valid_end = self.N - 2

    def __len__(self):
        return self.valid_end - self.valid_start + 1

    def __getitem__(self, idx):
        t = self.valid_start + idx

        # 1) Channel Past
        ch_list = []
        for k in range(t - self.past_len + 1, t + 1):
            coeffs_np = get_coeffs_from_frame(self.comm_frames[k], ue_idx=self.ue_idx)
            h = preprocess_channel_coeffs_minmax(
                coeffs_np,
                self.r_min, self.r_max, self.i_min, self.i_max,
                device=self.device
            ).reshape(-1)  # (2048,)
            ch_list.append(h)
        channel_past = torch.stack(ch_list, dim=0)  # (T, 2048)

        # 2) Target Next
        coeffs_np_next = get_coeffs_from_frame(self.comm_frames[t + 1], ue_idx=self.ue_idx)
        target = preprocess_channel_coeffs_minmax(
            coeffs_np_next,
            self.r_min, self.r_max, self.i_min, self.i_max,
            device=self.device
        ).reshape(-1)  # (2048,)

        return channel_past, target

# DataLoader 구현

In [8]:
comm_frames = flatten_comm_frames(dataset.comm_dataset)

ds = ChannelNextStepDatasetGPU(
    comm_frames=comm_frames,
    ue_idx=0,
    past_len=16,
    device=device
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=True,
    num_workers=0,     
    pin_memory=False   
)
ch, y = next(iter(loader))
print(ch.shape, y.shape)
print(ch.device, y.device)


torch.Size([8, 16, 2048]) torch.Size([8, 2048])
cuda:2 cuda:2


# Fine-tuning

In [9]:
class LSTMForecaster(nn.Module):
    """
    Input : ch  (B,T,F_in)
    Output: yhat(B,F_out)
    """
    def __init__(
        self,
        F_in: int,
        F_out: int,
        hidden: int = 256,
        num_layers: int = 2,
        dropout: float = 0.1,
        pool: str = "last",  # "last" or "mean"
    ):
        super().__init__()
        self.pool = pool

        # NOTE: dropout is applied only if num_layers > 1 in PyTorch LSTM
        self.lstm = nn.LSTM(
            input_size=F_in,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=(dropout if num_layers > 1 else 0.0),
        )
        self.head = nn.Linear(hidden, F_out)

    def forward(self, ch):
        # ch: (B,T,F_in)
        out, _ = self.lstm(ch)  # out: (B,T,hidden)

        if self.pool == "last":
            z = out[:, -1, :]     # (B,hidden)
        elif self.pool == "mean":
            z = out.mean(dim=1)   # (B,hidden)
        else:
            raise ValueError(f"Unknown pool={self.pool}")

        return self.head(z)       # (B,F_out)

# NMSE(dB)

In [10]:
@torch.no_grad()
def nmse_db(yhat: torch.Tensor, y: torch.Tensor, eps: float = 1e-16) -> torch.Tensor:
    # yhat, y: (B,F)
    num = torch.sum((yhat - y) ** 2, dim=1)
    den = torch.sum(y ** 2, dim=1).clamp_min(eps)
    nmse = num / den
    return 10.0 * torch.log10(nmse.clamp_min(eps)).mean()


# Train/val Split

In [11]:
# 겹치는 부분이 있는지 확인하는 함수
def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    for i in ds_indices:
        t = ds.valid_start + i
        # inputs use [t-T+1 .. t], target uses (t+1)
        used.update(range(t - T + 1, t + 2))
    return used

In [12]:
n_total = len(ds)
T = ds.past_len  # = past_len

# ---- choose k so that train=3k, val=k and NO frame-overlap is possible ----
k_max = (n_total - T) // 4
if k_max <= 0:
    raise ValueError(
        f"Not enough data for strict non-overlap 3:1. "
        f"n_total={n_total}, past_len(T)={T}. "
        f"Try increasing scenes or decreasing past_len."
    )

k = k_max               # use as much data as possible under constraints
n_val = k
n_train = 3 * k

train_idx = list(range(0, n_train))
val_idx   = list(range(n_total - n_val, n_total))

print("n_total:", n_total, "T:", T)
print("train samples:", len(train_idx), "val samples:", len(val_idx), "ratio:", len(train_idx)/len(val_idx))

# ---- min/max ONLY from train (same as your original policy) ----
train_ts = [ds.valid_start + i for i in train_idx]

(real_min, real_max), (imag_min, imag_max) = get_train_min_max_realimag(
    comm_frames, train_ts, us_idx=0
)

ds.r_min = real_min
ds.r_max = real_max
ds.i_min = imag_min
ds.i_max = imag_max

print("Dataset statistical values set in the dataset.")

train_ds = Subset(ds, train_idx)
val_ds   = Subset(ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

F_in = 2048


overlap = used_frames_for_ds_indices(ds, train_idx).intersection(
          used_frames_for_ds_indices(ds, val_idx))
print("frame-overlap count:", len(overlap))
assert len(overlap) == 0, "Leakage detected: train/val share frames!"

# Verify (channel-only)
ch, y = next(iter(train_loader))   # ✅ img 제거
F_out = y.shape[-1]

print("\n=== Data Check ===")
print(f"y stats | min: {y.min().item():.4f}, max: {y.max().item():.4f}")
print("If scaling worked correctly, values should be within [0, 1].")

# (선택) ch도 같이 범위 확인하고 싶으면
print(f"ch stats | min: {ch.min().item():.4f}, max: {ch.max().item():.4f}")

n_total: 984 T: 16
train samples: 726 val samples: 242 ratio: 3.0
Calculating min/max over training set...
Done. rmin=-1.599962900128591e-06, rmax=1.5977824378041814e-06, imin=-1.6400732538319291e-06, imax=1.579536450012739e-06
Dataset statistical values set in the dataset.
frame-overlap count: 0

=== Data Check ===
y stats | min: 0.1313, max: 0.8721
If scaling worked correctly, values should be within [0, 1].
ch stats | min: 0.0000, max: 1.0000


# Train/Eval 함수

In [13]:
# batch check
ch, y = next(iter(train_loader))
print(ch.shape, y.shape)

def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        yhat = model(ch)                
        loss = F.mse_loss(yhat, y)
        loss.backward()

        if grad_clip and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, y in loader:
        ch = ch.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        yhat = model(ch)                 
        loss = F.mse_loss(yhat, y)

        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)

torch.Size([32, 16, 2048]) torch.Size([32, 2048])


# 모델 생성 및 확인

In [14]:
# 배치 하나로 F_in/F_out 자동 확정
ch, y = next(iter(train_loader))
F_in  = ch.shape[-1]
F_out = y.shape[-1]
print("Detected:", "F_in=", F_in, "F_out=", F_out)
print("Batch devices:", ch.device, y.device)

model = LSTMForecaster(
    F_in=F_in,
    F_out=F_out,
    hidden=256,
    num_layers=2,   # ✅ 12 stacked LSTM layers
    dropout=0.1,
    pool="last",
).to(device)

# sanity forward
model.eval()
with torch.no_grad():
    # ds가 이미 cuda 텐서 반환이면 아래 .to(device) 생략 가능
    yhat = model(ch.to(device))
print("yhat:", yhat.shape, "y:", y.shape)



Detected: F_in= 2048 F_out= 2048
Batch devices: cuda:2 cuda:2


yhat: torch.Size([32, 2048]) y: torch.Size([32, 2048])


# Optimizer/Scheduler 설정

In [15]:
# requires_grad=True인 파라미터만 학습
trainable_params = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable_params))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# (선택) cosine scheduler
epochs = 500
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)



trainable params: 3414016


# 학습 루프 및 checkpoint 저장

In [16]:
import time
import os

best_val = float("inf")
best_val_nmse = None
best_epoch = None

t_train0 = time.time()  # ✅ 전체 학습 시작 시간

# 어떤 모델을 몇개의 scene으로 학습을 했는지 기록
log_path = "lstm_scene1000.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_nmse = train_one_epoch(model, train_loader, optimizer, device=device, grad_clip=1.0)
        va_loss, va_nmse = evaluate(model, val_loader, device=device)

        scheduler.step()

        dt = time.time() - t0

        # ✅ 1) print에 쓰는 문자열을 변수로 만들기
        line = (
            f"[{epoch:02d}/{epochs}] "
            f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
            f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
            f"{dt:.1f}s"
        )

        # ✅ 2) 콘솔 + 파일에 동시에 기록
        print(line)
        f.write(line + "\n")
        f.flush()

        if va_loss < best_val:
            best_val = va_loss
            best_val_nmse = va_nmse
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "F_in": F_in,
                    "F_out": F_out,
                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_val_nmse,
                },
                "bestc_channel_only_finetune.pt"
            )

            save_line = (
                f"  ↳ saved bestc_channel_only_finetune.pt | best@{best_epoch}: "
                f"val loss={best_val:.6f}, val nmse(dB)={best_val_nmse:.4f}"
            )

            print(save_line)
            f.write(save_line + "\n")
            f.flush()

# ✅ 전체 학습 종료 후 총 시간 출력
total_sec = time.time() - t_train0
h = int(total_sec // 3600)
m = int((total_sec % 3600) // 60)
s = total_sec % 60

summary1 = "\n=== Training Summary ==="
summary2 = f"Total time: {total_sec:.1f}s ({h}h {m}m {s:.1f}s)"
summary3 = f"Best epoch: {best_epoch} | best val loss={best_val:.6f}, best val nmse(dB)={best_val_nmse:.4f}"

print(summary1)
print(summary2)
print(summary3)

# ✅ 요약도 파일에 저장(append)
with open(log_path, "a", encoding="utf-8") as f:
    f.write(summary1 + "\n")
    f.write(summary2 + "\n")
    f.write(summary3 + "\n")

print("Log saved to:", os.path.abspath(log_path))

[01/500] train loss=0.210395, nmse(dB)=-1.2158 | val loss=0.122748, nmse(dB)=-3.4855 | 0.9s
  ↳ saved bestc_channel_only_finetune.pt | best@1: val loss=0.122748, val nmse(dB)=-3.4855
[02/500] train loss=0.066961, nmse(dB)=-6.5684 | val loss=0.031528, nmse(dB)=-9.8973 | 0.7s
  ↳ saved bestc_channel_only_finetune.pt | best@2: val loss=0.031528, val nmse(dB)=-9.8973
[03/500] train loss=0.021921, nmse(dB)=-11.7509 | val loss=0.020043, nmse(dB)=-12.7863 | 0.7s
  ↳ saved bestc_channel_only_finetune.pt | best@3: val loss=0.020043, val nmse(dB)=-12.7863
[04/500] train loss=0.017442, nmse(dB)=-13.2345 | val loss=0.018968, nmse(dB)=-13.2670 | 0.7s
  ↳ saved bestc_channel_only_finetune.pt | best@4: val loss=0.018968, val nmse(dB)=-13.2670
[05/500] train loss=0.017027, nmse(dB)=-13.4301 | val loss=0.018913, nmse(dB)=-13.3071 | 0.7s
  ↳ saved bestc_channel_only_finetune.pt | best@5: val loss=0.018913, val nmse(dB)=-13.3071
[06/500] train loss=0.017066, nmse(dB)=-13.4384 | val loss=0.018939, nmse(dB

# 런타임 체크

In [25]:
ch, y = next(iter(train_loader))
print("y abs mean:", y.abs().mean().item())
print("y abs max :", y.abs().max().item())
print("y power   :", (y**2).mean().item())

with torch.no_grad():
    yhat = model(ch.to(device))   
print("yhat abs mean:", yhat.abs().mean().item())
print("yhat abs max :", yhat.abs().max().item())
print("yhat power   :", (yhat**2).mean().item())

y abs mean: 0.506099283695221
y abs max : 0.860482394695282
y power   : 0.27246594429016113
yhat abs mean: 0.5073773860931396
yhat abs max : 0.8739141821861267
yhat power   : 0.2630261182785034


In [26]:
y.shape

torch.Size([32, 2048])

In [27]:
ch.shape

torch.Size([32, 16, 2048])

In [28]:
y[0:10, :10]

tensor([[0.4368, 0.4447, 0.4550, 0.4667, 0.4785, 0.4900, 0.5019, 0.5149, 0.5285,
         0.5410],
        [0.4742, 0.4856, 0.4971, 0.5088, 0.5204, 0.5320, 0.5434, 0.5545, 0.5650,
         0.5749],
        [0.7147, 0.7177, 0.7203, 0.7216, 0.7209, 0.7177, 0.7125, 0.7057, 0.6983,
         0.6908],
        [0.3650, 0.3717, 0.3792, 0.3874, 0.3963, 0.4057, 0.4155, 0.4256, 0.4357,
         0.4457],
        [0.4433, 0.4476, 0.4526, 0.4597, 0.4691, 0.4799, 0.4904, 0.4999, 0.5090,
         0.5187],
        [0.4102, 0.4166, 0.4221, 0.4267, 0.4303, 0.4331, 0.4357, 0.4386, 0.4423,
         0.4473],
        [0.6871, 0.7017, 0.7153, 0.7265, 0.7350, 0.7411, 0.7458, 0.7505, 0.7561,
         0.7631],
        [0.6170, 0.6040, 0.5883, 0.5708, 0.5537, 0.5393, 0.5280, 0.5184, 0.5082,
         0.4963],
        [0.6362, 0.6271, 0.6177, 0.6082, 0.5986, 0.5889, 0.5794, 0.5699, 0.5607,
         0.5516],
        [0.5283, 0.5397, 0.5495, 0.5580, 0.5662, 0.5750, 0.5849, 0.5959, 0.6075,
         0.6186]], device='c

In [ ]:
yhat[0:10, :10]

tensor([[0.4914, 0.4919, 0.4836, 0.4985, 0.4785, 0.4966, 0.4968, 0.5045, 0.4963,
         0.5052],
        [0.5762, 0.5854, 0.5788, 0.6155, 0.5983, 0.6106, 0.6073, 0.5801, 0.6287,
         0.6235],
        [0.5031, 0.5162, 0.5105, 0.4904, 0.4997, 0.5080, 0.5076, 0.5135, 0.5037,
         0.5074],
        [0.4351, 0.4248, 0.4229, 0.4484, 0.4183, 0.4590, 0.4654, 0.4822, 0.4795,
         0.4905],
        [0.5231, 0.5203, 0.5243, 0.5191, 0.5229, 0.5240, 0.5203, 0.5206, 0.5271,
         0.5218],
        [0.5095, 0.5114, 0.5073, 0.5103, 0.5071, 0.5075, 0.5165, 0.5047, 0.5117,
         0.5122],
        [0.6745, 0.6449, 0.6612, 0.6694, 0.6819, 0.6947, 0.6833, 0.7490, 0.6608,
         0.7228],
        [0.4920, 0.5057, 0.4977, 0.4970, 0.4825, 0.4987, 0.5011, 0.4907, 0.5023,
         0.5066],
        [0.5293, 0.5118, 0.5240, 0.5264, 0.5605, 0.5823, 0.5311, 0.5585, 0.5471,
         0.5921],
        [0.5318, 0.5338, 0.5376, 0.5398, 0.5193, 0.5387, 0.5505, 0.5506, 0.5354,
         0.5483]], device='c

: 

# 데이터 입력 형태

In [22]:
print("=== dataset sizes ===")
print("N(comm_frames):", len(comm_frames))
print("past_len      :", ds.past_len)
print("len(ds)       :", len(ds))
print("len(train_ds) :", len(train_ds))
print("len(val_ds)   :", len(val_ds))
print("len(train_loader):", len(train_loader))
print("len(val_loader)  :", len(val_loader))

print("\n=== one batch shapes ===")
ch, y = next(iter(train_loader))
print("ch :", tuple(ch.shape), " -> (B,T,F_in)")
print("y  :", tuple(y.shape), " -> (B,F_out)")
with torch.no_grad():
    yhat = model(ch.to(device))
print("yhat:", tuple(yhat.shape), " -> (B,F_out)")
print("this forward predicted vectors:", yhat.shape[0], "(=B)")
print("each vector predicts elements:", yhat.shape[1], "(=F_out)")

=== dataset sizes ===
N(comm_frames): 1000
past_len      : 16
len(ds)       : 984
len(train_ds) : 726
len(val_ds)   : 242
len(train_loader): 23
len(val_loader)  : 8

=== one batch shapes ===
ch : (32, 16, 2048)  -> (B,T,F_in)
y  : (32, 2048)  -> (B,F_out)
yhat: (32, 2048)  -> (B,F_out)
this forward predicted vectors: 32 (=B)
each vector predicts elements: 2048 (=F_out)
